In [1]:
import pandas as pd
import numpy as np

In [2]:
billing_df = pd.read_csv("Billing_Data.csv")
patient_df = pd.read_csv("Patient_Data.csv")

In [3]:
display(patient_df.head())
display(billing_df.head())

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45


,PatientID,InsuranceCovered,FinalAmount
0,101,2000,3000
1,102,1500,3500
2,103,2500,5000
3,104,3000,3200
4,105,1000,4000


In [4]:
# 1. Load the patient dataset and show summary with info().
display(patient_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


None

In [5]:
display(billing_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   PatientID         5 non-null      int64
 1   InsuranceCovered  5 non-null      int64
 2   FinalAmount       5 non-null      int64
dtypes: int64(3)
memory usage: 252.0 bytes


None

In [6]:
# 2. Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].

billing = patient_df[["PatientID", "Department", "Doctor", "BillAmount"]]
billing

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN
5,101,Cardiology,Dr. Smith,5000.0


In [7]:
# 3. Drop administrative columns like ['ReceptionistID', 'CheckInTime'].

patient_df = patient_df.drop(["ReceptionistID", "CheckInTime"], axis = 1)
patient_df

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN
5,101,Alice,Cardiology,Dr. Smith,5000.0


In [8]:
# 4. se groupby to find total bill amount per department.

bill_department = patient_df.groupby("Department")["BillAmount"].sum()
bill_department

Department
Cardiology     16200.0
Dermatology        0.0
Neurology          0.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64

In [9]:
# 5. Remove duplicate patient records based on PatientID.

patient_df.duplicated(subset=['PatientID'])

0    False
1    False
2    False
3    False
4    False
5     True
dtype: bool

In [10]:
patient_df.drop_duplicates(subset=['PatientID'], inplace = True)

In [11]:
patient_df

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN


In [12]:
# 6. Fill missing BillAmount values with the mean bill amount.

patient_df["BillAmount"].isnull().sum()

np.int64(2)

In [13]:
patient_df["BillAmount"].fillna(patient_df["BillAmount"].mean(), inplace=True)

C:\Users\Basavaraj M\AppData\Local\Temp\ipykernel_4068\2897431026.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  patient_df["BillAmount"].fillna(patient_df["BillAmount"].mean(), inplace=True)


In [14]:
patient_df

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000
1,102,Bob,Neurology,Dr. John,6233.333333
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000
3,104,David,Cardiology,Dr. Smith,6200.000000
4,105,Eva,Dermatology,Dr. Rose,6233.333333


In [15]:
# 7. Merge the billing dataset with patient dataset on PatientID.

merged_df = pd.merge(patient_df, billing_df, on="PatientID")
merged_df

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000,3000
1,102,Bob,Neurology,Dr. John,6233.333333,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.000000,3000,3200
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000,4000


In [16]:
# 8. Concatenate an additional DataFrame that contains new patients for the current week (row-wise).

data = [
    {"PatientID":111, "Name":"Mohan", "Department":"Neurology", "Doctor":"Dr.ranade", "BillAmount":50000},
    {"PatientID":112, "Name":"Jaggu", "Department":"Dermatology", "Doctor":"Dr.ranade", "BillAmount":230000}
     ]

In [17]:
new_df = pd.DataFrame(data)
new_df

,PatientID,Name,Department,Doctor,BillAmount
0,111,Mohan,Neurology,Dr.ranade,50000
1,112,Jaggu,Dermatology,Dr.ranade,230000


In [18]:
final_df = pd.concat([merged_df,new_df], axis=0, ignore_index=True)

In [19]:
final_df

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000.0,3000.0
1,102,Bob,Neurology,Dr. John,6233.333333,1500.0,3500.0
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500.0,5000.0
3,104,David,Cardiology,Dr. Smith,6200.000000,3000.0,3200.0
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000.0,4000.0
5,111,Mohan,Neurology,Dr.ranade,50000.000000,NaN,NaN
6,112,Jaggu,Dermatology,Dr.ranade,230000.000000,NaN,NaN


In [20]:
# 9. Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).

bill_data = [
    {"InsuranceCovered":12300, "FinalAmount":37700},
    {"InsuranceCovered":10000, "FinalAmount":220000}
        ]

In [21]:
bill_df = pd.DataFrame(bill_data)

In [28]:
# Keep first 5 rows as they are
old_rows = final_df.iloc[:5]
old_rows

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000.0,3000.0
1,102,Bob,Neurology,Dr. John,6233.333333,1500.0,3500.0
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500.0,5000.0
3,104,David,Cardiology,Dr. Smith,6200.000000,3000.0,3200.0
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000.0,4000.0


In [25]:
# Take last 2 rows without the NaN columns
new_rows = final_df.iloc[5:, :5]
new_rows

,PatientID,Name,Department,Doctor,BillAmount
5,111,Mohan,Neurology,Dr.ranade,50000.0
6,112,Jaggu,Dermatology,Dr.ranade,230000.0


In [26]:
# Column wise concatenation
new_rows = pd.concat([new_rows.reset_index(drop=True),
                      bill_df.reset_index(drop=True)], axis=1)
new_rows

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,111,Mohan,Neurology,Dr.ranade,50000.0,12300,37700
1,112,Jaggu,Dermatology,Dr.ranade,230000.0,10000,220000


In [27]:
# Combine back all rows
final_df = pd.concat([old_rows, new_rows], ignore_index=True)
final_df

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000.0,3000.0
1,102,Bob,Neurology,Dr. John,6233.333333,1500.0,3500.0
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500.0,5000.0
3,104,David,Cardiology,Dr. Smith,6200.000000,3000.0,3200.0
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000.0,4000.0
5,111,Mohan,Neurology,Dr.ranade,50000.000000,12300.0,37700.0
6,112,Jaggu,Dermatology,Dr.ranade,230000.000000,10000.0,220000.0
